In [2]:
import pandas as pd
import numpy as np
from typing import Optional, Union

df = pd.read_parquet("train.parquet")


def preprocess_column(
    df: pd.DataFrame,
    col: str,

    # --- Missing values ---
    delete_empty_entries: bool = False,       # Drop rows where col is NaN
    replace_by_avg: bool = False,             # Fill NaN with column mean
    replace_by_median: bool = False,          # Fill NaN with column median
    replace_by_mode: bool = False,            # Fill NaN with column mode (categorical-friendly)
    fill_value: Optional[Union[int, float, str]] = None,  # Fill NaN with a custom constant

    # --- Outlier handling ---
    # There are some spikes that you might want to treat as outliers
    clip_outliers_iqr: bool = False,          # Clip to [Q1 - 1.5*IQR, Q3 + 1.5*IQR]
    clip_outliers_percentile: Optional[tuple] = None,  # e.g. (0.01, 0.99)

    # --- Transformations ---
    log1p_transform: bool = False,            # np.log1p — for right-skewed, non-negative cols
    normalize_minmax: bool = False,           # Scale to [0, 1]
    standardize: bool = False,                # Z-score: mean=0, std=1

    # --- Type casting ---
    cast_to: Optional[str] = None,            # "float", "int", "str", "category"

    # --- Utility ---
    drop_duplicates: bool = False,            # Drop rows with duplicate values in this col
    inplace: bool = False,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    One-stop preprocessing function for a single DataFrame column.
    Steps run in this order:
      1. ±inf handling
      2. NaN handling
      3. Outlier clipping
      4. Transformation (log1p → scale/normalize)
      5. Type cast
      6. Deduplication
    """

    if col not in df.columns:
        raise ValueError(f"Column '{col}' not found. Available: {list(df.columns)}")

    df = df if inplace else df.copy()
    log = []


    # ── 2. Missing values ────────────────────────────────────────────────────
    n_nan = df[col].isna().sum()

    if delete_empty_entries:
        df = df.dropna(subset=[col]).reset_index(drop=True)
        log.append(f"delete_empty_entries       → dropped {n_nan} rows with NaN")

    elif replace_by_avg:
        val = df[col].mean()
        df[col] = df[col].fillna(val)
        log.append(f"replace_by_avg             → filled {n_nan} NaN with mean = {val:.4f}")

    elif replace_by_median:
        val = df[col].median()
        df[col] = df[col].fillna(val)
        log.append(f"replace_by_median          → filled {n_nan} NaN with median = {val:.4f}")

    elif replace_by_mode:
        val = df[col].mode()[0]
        df[col] = df[col].fillna(val)
        log.append(f"replace_by_mode            → filled {n_nan} NaN with mode = {val!r}")

    elif fill_value is not None:
        df[col] = df[col].fillna(fill_value)
        log.append(f"fill_value                 → filled {n_nan} NaN with {fill_value!r}")

    # ── 3. Outlier clipping ──────────────────────────────────────────────────
    if clip_outliers_iqr:
        q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n_clip = ((df[col] < lower) | (df[col] > upper)).sum()
        df[col] = df[col].clip(lower=lower, upper=upper)
        log.append(f"clip_outliers_iqr          → clipped {n_clip} values to [{lower:.4f}, {upper:.4f}]")

    elif clip_outliers_percentile is not None:
        lo, hi = clip_outliers_percentile
        lower, upper = df[col].quantile(lo), df[col].quantile(hi)
        n_clip = ((df[col] < lower) | (df[col] > upper)).sum()
        df[col] = df[col].clip(lower=lower, upper=upper)
        log.append(f"clip_outliers_percentile   → clipped {n_clip} values to [{lower:.4f}, {upper:.4f}] (p{lo*100:.0f}–p{hi*100:.0f})")

    # ── 4. Transformations ───────────────────────────────────────────────────
    if log1p_transform:
        if (df[col].dropna() < 0).any():
            raise ValueError(f"log1p_transform requires non-negative values in '{col}'.")
        df[col] = np.log1p(df[col])
        log.append("log1p_transform            → applied np.log1p(x)")

    if normalize_minmax:
        col_min, col_max = df[col].min(), df[col].max()
        df[col] = (df[col] - col_min) / (col_max - col_min)
        log.append(f"normalize_minmax           → scaled to [0, 1] (was [{col_min:.4f}, {col_max:.4f}])")

    elif standardize:
        mu, sigma = df[col].mean(), df[col].std()
        df[col] = (df[col] - mu) / sigma
        log.append(f"standardize                → z-score (μ={mu:.4f}, σ={sigma:.4f})")

    # ── 5. Type cast ─────────────────────────────────────────────────────────
    if cast_to is not None:
        df[col] = df[col].astype(cast_to)
        log.append(f"cast_to                    → dtype cast to '{cast_to}'")

    # ── 6. Deduplication ─────────────────────────────────────────────────────
    if drop_duplicates:
        n_before = len(df)
        df = df.drop_duplicates(subset=[col]).reset_index(drop=True)
        log.append(f"drop_duplicates            → removed {n_before - len(df)} rows")

    # ── Verbose summary ──────────────────────────────────────────────────────
    if verbose:
        sep = "=" * 57
        print(f"\n{sep}\n  preprocess_column  >>>  '{col}'\n{sep}")
        for step in log or ["  (no operations applied)"]:
            print(f"  ✓ {step}")
        print(f"  {'─'*51}")
        print(f"  Final shape : {df.shape}  |  NaN left: {df[col].isna().sum()}  |  dtype: {df[col].dtype}")
        print(f"{sep}\n")

    return df

In [3]:
# Example of usage

df = preprocess_column(df, col="feature_ag",
    replace_by_avg = True)


  preprocess_column  >>>  'feature_ag'
  ✓ replace_by_avg             → filled 8126 NaN with mean = 1.2337
  ───────────────────────────────────────────────────
  Final shape : (5337414, 94)  |  NaN left: 0  |  dtype: float64

